In [16]:
# Department: ESTSOFT
# Class: AI Modelling
# Category: Deep Learing, Game
# Title: TellingByfAIce
# Contributors: Kimm Soo Min
# Last modified date: 21/06/25

In [17]:
# Library
import os
import json
from tqdm import tqdm
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

In [18]:
# Device agnostic code
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [19]:
# Path
data_path = "/workspace/data"

In [20]:
# Hyperparameters
label_map = {"anger": 0, "happy": 1, "panic": 2, "sadness": 3}
num_labels = len(label_map)
image_size = 224
batch_size = 32
epochs = 100

In [21]:
# Load json
def load_json(file):
	with open(file=file, mode='r', encoding='euc-kr') as f:
		return json.load(f)

In [22]:
# Dataset - Detector
class DetectorDataset(Dataset):
	def __init__(self, dir_root, set_name, transform):
		self.samples = []
		self.dir_root = dir_root
		self.set_name = set_name  # train, val, test
		self.transform = transform

		# Set paths
		if set_name == "test":
			label_dir = os.path.join(dir_root, "test", "label")
			img_dir_root = os.path.join(dir_root, "test", "image")
		else:
			label_dir = os.path.join(dir_root, "train", "label", set_name)
			img_dir_root = os.path.join(dir_root, "train", "image", set_name)

		# Iterate through the directory to grab information about images
		label_paths = [os.path.join(label_dir, f) for f in os.listdir(label_dir)]
		for label_path in label_paths:
			label = os.path.basename(label_path).split("_")[-1].split(".")[0]  # "train_anger.json" -> "anger"
			img_dir = os.path.join(img_dir_root, label)

			data = load_json(label_path)
			for item in data:
				image_path = os.path.join(img_dir, item["filename"])

				# Check if the file actually exists
				if not os.path.exists(image_path):
					continue

				self.samples.append({"image_path": image_path,
									 "bbox": item["annot_A"]["boxes"]})

	def __len__(self):
		return len(self.samples)

	def __getitem__(self, idx):
		item = self.samples[idx]

		image = Image.open(item["image_path"]).convert("RGB")
		image = self.transform(image) # C, W, H
		width, height = image.shape[2], image.shape[1] # Required for normalisation

		bbox = item["bbox"]
		X_min = bbox["minX"] / width
		Y_min = bbox["minY"] / height
		X_max = bbox["maxX"] / width
		Y_max = bbox["maxY"] / height
		bbox = torch.tensor([X_min, Y_min, X_max, Y_max], dtype=torch.float32)

		return image, bbox

In [23]:
# Transform
transform = transforms.Compose([transforms.Resize((image_size, image_size)),
								transforms.ToTensor()])

In [24]:
# Dataset
ds_train = DetectorDataset(dir_root=data_path, set_name="train", transform=transform)
ds_val   = DetectorDataset(dir_root=data_path, set_name="val", transform=transform)
ds_test  = DetectorDataset(dir_root=data_path, set_name="test", transform=transform)

In [25]:
# Dataloader
dl_train = DataLoader(dataset=ds_train, batch_size=batch_size, shuffle=True)
dl_val = DataLoader(dataset=ds_val, batch_size=batch_size, shuffle=False)
dl_test = DataLoader(dataset=ds_test, batch_size=batch_size, shuffle=False)

In [26]:
# ViT - Detector
class vit_detector(nn.Module):
	def __init__(self, num_classes):
		super().__init__()
		self.backbone = timm.create_model("vit_base_patch16_224", pretrained=True)
		self.num_features = self.backbone.head.in_features
		self.backbone.head = nn.Identity()
		self.fc_bbox = nn.Linear(self.num_features, num_classes)

	def forward(self, x):
		z = self.backbone(x)
		return self.fc_bbox(z)

In [27]:
# Instantiation
model_detector = vit_detector(num_classes=4).to(device)
optimizer_detector = torch.optim.Adam(model_detector.parameters(), lr=1e-5)
loss_detector = nn.SmoothL1Loss()

In [28]:
# Early stopping
loss_val_best = float('inf')
patience = 5          
patience_counter = 0 

In [29]:
losses_train = []
losses_val = []

for epoch in tqdm(range(epochs)):
	# Training
	model_detector.train()
	loss_train_total = 0
	for X_train, y_train in dl_train:
		X_train, y_train = X_train.to(device), y_train.to(device)

		y_train_pred = model_detector(X_train)
		loss_train = loss_detector(y_train_pred, y_train)

		optimizer_detector.zero_grad()
		loss_train.backward()
		optimizer_detector.step()

		loss_train_total += loss_train.item()

	losses_train.append(loss_train_total / len(dl_train))

	# Validation
	model_detector.eval()
	loss_val_total = 0
	with torch.inference_mode():
		for X_val, y_val in dl_val:
			X_val, y_val = X_val.to(device), y_val.to(device)

			y_val_pred = model_detector(X_val)
			loss_val = loss_detector(y_val_pred, y_val)
			
			loss_val_total += loss_val.item()
		
		loss_val_avg = loss_val_total / len(dl_val)
		losses_val.append(loss_val_avg)

	print(f"Epoch {epoch + 1} | Train Loss: {loss_train_total / len(dl_train):.4f} | Val Loss: {loss_val_total / len(dl_val):.4f}")
	
	# Early stopping
	if loss_val_avg < loss_val_best:
		loss_val_best = loss_val_avg
		patience_counter = 0
		torch.save(model_detector.state_dict(), f"/workspace/experiment/vit_detector_best.pth")
	else:
		patience_counter += 1
		if patience_counter >= patience:
			print(f"Early stopping triggered at epoch {epoch + 1}")
			break

	# Save periodically
	if (epoch + 1) % 5 == 0:
		torch.save(model_detector.state_dict(), f"/workspace/experiment/vit_detector_epoch_{epoch+1}.pth")

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.7829 | Val Loss: 0.5735


  1%|          | 1/100 [11:35<19:07:49, 695.65s/it]

Epoch 2 | Train Loss: 0.4242 | Val Loss: 0.5294


  2%|▏         | 2/100 [23:14<18:59:27, 697.63s/it]

...

Epoch 32 | Train Loss: 0.0145 | Val Loss: 0.4626


 33%|███▎      | 33/100 [5:22:45<10:47:36, 579.94s/it]

Epoch 33 | Train Loss: 0.0141 | Val Loss: 0.4756


 33%|███▎      | 33/100 [5:32:04<11:14:12, 603.77s/it]

Epoch 34 | Train Loss: 0.0135 | Val Loss: 0.4651
Early stopping triggered at epoch 34


In [31]:
# Testing
model_detector.eval()
losses_test = []
loss_test_total = 0
with torch.inference_mode():
	for X_test, y_test in dl_test:
		X_test, y_test = X_test.to(device), y_test.to(device)

		y_test_pred  = model_detector(X_test)
		loss_test = loss_detector(y_test_pred, y_test)

		loss_test_total += loss_test.item()
		
	losses_test.append(loss_test_total / len(dl_test))

print(f"\nTest Loss: {loss_test_total / len(dl_test):.4f}")


Test Loss: 0.4348
